
# ex2 - Self-training 半監督學習信用卡詐欺偵測

本 Notebook 使用 Self-training 方法結合監督與非監督學習流程：

1. 使用部分標記資料訓練初始模型（XGBoost）
2. 預測未標記資料，挑選高信心樣本加入訓練資料
3. 迭代訓練模型以提升對詐欺樣本的辨識能力


In [2]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
import xgboost as xgb
import kagglehub


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

def evaluation(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f'\n{model_name} Evaluation:')
    print('===' * 15)
    print('         Accuracy:', accuracy)
    print('  Precision Score:', precision)
    print('     Recall Score:', recall)
    print('         F1 Score:', f1)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))


In [4]:

RANDOM_SEED = 42
TEST_SIZE = 0.3

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
data = pd.read_csv(f"{path}/creditcard.csv")
data['Class'] = data['Class'].astype(int)

# 前處理
data.drop(columns=['Time'], inplace=True)
data['Amount'] = StandardScaler().fit_transform(data['Amount'].values.reshape(-1, 1))

X = data.drop(columns=['Class']).values
Y = data['Class'].values


In [5]:

# 隨機抽出少部分資料保留標籤，其餘設為 -1 (未標記)
X_train, X_test, y_train_true, y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, stratify=Y, random_state=RANDOM_SEED)

y_train = np.copy(y_train_true)
mask = np.random.rand(len(y_train)) < 0.95  # 95% 設為未標記
y_train[mask] = -1


In [6]:

xgb_model = xgb.XGBClassifier(
    n_estimators=200, use_label_encoder=False, eval_metric="logloss",
    random_state=RANDOM_SEED, scale_pos_weight=100
)

self_training_model = SelfTrainingClassifier(
    base_estimator=xgb_model, threshold=0.9, verbose=True
)

self_training_model.fit(X_train, y_train)
y_pred = self_training_model.predict(X_test)

evaluation(y_test, y_pred, model_name="Self-training XGBoost")


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\sklearn\semi_supervised\_self_training.py:210: FutureWarning: `base_estimator` has been deprecated in 1.6 and will be removed in 1.8. Please use `estimator` instead.
  warn(
c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


End of iteration 1, added 189336 new labels.


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


End of iteration 2, added 64 new labels.


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


...

End of iteration 5, added 1 new labels.


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


End of iteration 6, added 1 new labels.


c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\MH\NTCUcollege\四下\NTCU-Machine-Learning\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [15:23:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Self-training XGBoost Evaluation:
         Accuracy: 0.9993445923013002
  Precision Score: 0.8650793650793651
     Recall Score: 0.7364864864864865
         F1 Score: 0.7956204379562044

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.87      0.74      0.80       148

    accuracy                           1.00     85443
   macro avg       0.93      0.87      0.90     85443
weighted avg       1.00      1.00      1.00     85443

